# 🚗 Pothole Detection — YOLOv11 Training on Google Colab (Fresh Start)
**Dataset:** RDD2022 (Road Damage Dataset) via public Kaggle repository 
**Model:** YOLOv11s 
**GPU:** Free T4 (Colab) 

---
### ⚡ Steps:
1. Check GPU
2. Install Ultralytics
3. Download Dataset (Unrestricted public Kaggle repository)
4. Extract Dataset
5. Create a smaller Subset (Optional — for fast training)
6. Generate YAML Config
7. Verify Dataset Counts
8. Train YOLOv11s Model
9. Save / Download Weights

## ✅ Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ WARNING: GPU is not active. Please go to Runtime -> Change runtime type -> select T4 GPU.')

## ✅ Step 2: Install Ultralytics (YOLOv11)

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
print('Ultralytics installed successfully!')

## ✅ Step 3: Configure Kaggle API & Download Dataset
Downloads the road damage dataset from an unrestricted repository to bypass `403 Forbidden` issues.

In [ ]:
# Configure Kaggle credentials automatically
import os
import json

os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_creds = {
    "username": "utkarshsahay1509",
    "key": "KGAT_c99494e135f62f3afbe67a626b8c840c"
}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle API configured successfully with credentials!')

# Download dataset
!pip install kaggle -q
print('Downloading RDD2022 dataset from public repository...')
!kaggle datasets download -d aliabdelmenam/rdd-2022 -p /content/rdd_data
print('✅ Download complete!')

## ✅ Step 4: Extract Dataset

In [ ]:
import zipfile, os

ZIP_PATH = '/content/rdd_data'
EXTRACT_PATH = '/content/rdd_data/extracted'

# Ensure directory exists
os.makedirs(ZIP_PATH, exist_ok=True)

# Find and extract zip
zip_files = [f for f in os.listdir(ZIP_PATH) if f.endswith('.zip')]

if not zip_files:
    print(f"❌ ERROR: No .zip files found in '{ZIP_PATH}'!")
    print("Please check if Step 3 succeeded.")
else:
    for f in zip_files:
        zip_file = os.path.join(ZIP_PATH, f)
        print(f'Extracting: {f}')
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(EXTRACT_PATH)
        print('✅ Extraction done!')
        break

    # List extracted contents
    print('\n--- Extracted Folder Structure ---')
    for root, dirs, files in os.walk(EXTRACT_PATH):
        level = root.replace(EXTRACT_PATH, '').count(os.sep)
        if level < 2:
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')

## ⚡ Step 5: (OPTIONAL) Create Dataset Subset for Fast Training
The full dataset has ~27,000 images, which can take several hours to train. Run this cell to randomly select **2500 training images and 500 validation images** (total 3000 images) and train on it instead. This will reduce training time to a few minutes!

In [ ]:
# Create a smaller subset of dataset to speed up training
import os
import random
import shutil

# Base directory where dataset was extracted
base_dir = '/content/rdd_data/extracted'

# Dynamically search for a folder containing 'train/images'
ORIGINAL_DATA_PATH = None
for root, dirs, files in os.walk(base_dir):
    if root.endswith('train/images') or root.endswith('train\\images'):
        ORIGINAL_DATA_PATH = os.path.dirname(os.path.dirname(root))
        break

SUBSET_DATA_PATH = '/content/rdd_subset'

if ORIGINAL_DATA_PATH is None:
    print("❌ ERROR: Original dataset 'train/images' not found. Please ensure extraction succeeded.")
else:
    # Create subset directories
    for split in ['train', 'val']:
        os.makedirs(os.path.join(SUBSET_DATA_PATH, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(SUBSET_DATA_PATH, split, 'labels'), exist_ok=True)

    # Function to copy subset
    def create_subset(split, target_size):
        img_dir = os.path.join(ORIGINAL_DATA_PATH, split, 'images')
        lbl_dir = os.path.join(ORIGINAL_DATA_PATH, split, 'labels')
        
        all_images = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"Original {split} images: {len(all_images)}")
        
        # Randomly select a subset
        subset_images = random.sample(all_images, min(target_size, len(all_images)))
        print(f"Selected {len(subset_images)} images for subset {split}")
        
        copied_count = 0
        for img_name in subset_images:
            img_src = os.path.join(img_dir, img_name)
            lbl_src = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + '.txt')
            
            img_dst = os.path.join(SUBSET_DATA_PATH, split, 'images', img_name)
            shutil.copy2(img_src, img_dst)
            
            if os.path.exists(lbl_src):
                lbl_dst = os.path.join(SUBSET_DATA_PATH, split, 'labels', os.path.splitext(img_name)[0] + '.txt')
                shutil.copy2(lbl_src, lbl_dst)
            copied_count += 1
                
        print(f"Successfully copied {copied_count} {split} images and labels.")

    # Select 2500 training images and 500 validation images
    random.seed(42)
    create_subset('train', 2500)
    create_subset('val', 500)
    print(f"✅ Downsampled subset generated at: {SUBSET_DATA_PATH}")

## ✅ Step 6: Create Dataset YAML Config

In [ ]:
import os

# Prefer the downsampled subset if it exists, otherwise fall back to search
if os.path.exists('/content/rdd_subset/train/images'):
    DATASET_PATH = '/content/rdd_subset'
    print(f"✅ Using downsampled subset: {DATASET_PATH}")
else:
    # Base directory where dataset was extracted
    base_dir = '/content/rdd_data/extracted'
    
    # Dynamically search for a folder containing 'train/images'
    DATASET_PATH = None
    for root, dirs, files in os.walk(base_dir):
        if root.endswith('train/images') or root.endswith('train\\images'):
            DATASET_PATH = os.path.dirname(os.path.dirname(root))
            break
    
    # Fallback: Check Google Drive
    if DATASET_PATH is None:
        drive_base = '/content/drive/MyDrive'
        if os.path.exists(drive_base):
            for root, dirs, files in os.walk(drive_base):
                if root.endswith('train/images') or root.endswith('train\\images'):
                    DATASET_PATH = os.path.dirname(os.path.dirname(root))
                    break

if DATASET_PATH is None:
    DATASET_PATH = '/content/rdd_data/extracted/RDD_SPLIT'  # Default fallback
    print("⚠️ WARNING: Could not find any directory containing 'train/images'. Using default path.")
else:
    if DATASET_PATH != '/content/rdd_subset':
        print(f"✅ Success! Dynamically found dataset path: {DATASET_PATH}")

yaml_content = f"""
path: {DATASET_PATH}
train: train/images
val: val/images
test: test/images

names:
  0: Longitudinal Crack
  1: Transverse Crack
  2: Alligator Crack
  3: Other corruption
  4: Pothole
"""

YAML_PATH = '/content/pothole_data.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content.strip())

print(f'YAML saved to {YAML_PATH}')
print(yaml_content)

## ✅ Step 7: Verify Dataset Counts

In [ ]:
# Count images to verify dataset counts
import os

# Prefer the downsampled subset if it exists, otherwise fall back to search
if os.path.exists('/content/rdd_subset/train/images'):
    DATASET_PATH = '/content/rdd_subset'
else:
    base_dir = '/content/rdd_data/extracted'
    DATASET_PATH = None
    for root, dirs, files in os.walk(base_dir):
        if root.endswith('train/images') or root.endswith('train\\images'):
            DATASET_PATH = os.path.dirname(os.path.dirname(root))
            break
    
    if DATASET_PATH is None:
        drive_base = '/content/drive/MyDrive'
        if os.path.exists(drive_base):
            for root, dirs, files in os.walk(drive_base):
                if root.endswith('train/images') or root.endswith('train\\images'):
                    DATASET_PATH = os.path.dirname(os.path.dirname(root))
                    break

if DATASET_PATH is None:
    DATASET_PATH = '/content/rdd_data/extracted/RDD_SPLIT'

train_path = os.path.join(DATASET_PATH, 'train', 'images')
val_path   = os.path.join(DATASET_PATH, 'val',   'images')

if os.path.exists(train_path):
    print(f'✅ Dataset folder verified at: {DATASET_PATH}')
    print(f'Train images: {len(os.listdir(train_path))}')
    print(f'Val   images: {len(os.listdir(val_path))}')
else:
    print(f"❌ ERROR: 'train/images' directory not found inside '{DATASET_PATH}'!")
    print("Please verify if the extraction cell ran successfully.")

## ✅ Step 8: Train the Model
Trains YOLOv11s on our dataset (or subset).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')  # Auto-downloads weights

results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,          # T4 has 16GB VRAM — use batch 16
    patience=30,
    save=True,
    device='0',
    amp=True,
    workers=2,
    name='pothole_colab_v1',
    cos_lr=True,
    project='/content/runs'
)

print('✅ Training Complete!')

## ✅ Step 9: Save Results to Google Drive & Download Weights

In [ ]:
# Mount Drive and copy trained model
from google.colab import drive
drive.mount('/content/drive')

import shutil
BEST_MODEL = '/content/runs/pothole_colab_v1/weights/best.pt'
DRIVE_SAVE  = '/content/drive/MyDrive/pothole_best.pt'

try:
    shutil.copy(BEST_MODEL, DRIVE_SAVE)
    print(f'✅ Model saved successfully to Google Drive: {DRIVE_SAVE}')
except Exception as e:
    print(f'⚠️ Error saving to Drive (Make sure Drive is mounted): {e}')

# Trigger direct download to PC
from google.colab import files
try:
    files.download('/content/runs/pothole_colab_v1/weights/best.pt')
    print('Direct download triggered!')
except Exception as e:
    print(f'Direct download not available: {e}')

## ✅ Step 10: Quick Inference Test (Optional)
Run inference on a random validation image to verify the model is working.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os, random

best_model_path = '/content/runs/pothole_colab_v1/weights/best.pt'
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    val_dir = os.path.join(DATASET_PATH, 'val/images')
    sample = random.choice(os.listdir(val_dir))
    img_path = os.path.join(val_dir, sample)

    results = model.predict(img_path, conf=0.3, save=True, project='/content/test_out')
    out_img = f'/content/test_out/predict/{sample}'
    display(Image(out_img))
else:
    print('❌ Trained weights not found. Did training complete?')